In [1]:
import os
import json
import random
import math
import numpy as np
import cv2
import gradio as gr
from tqdm import tqdm

# -------------------------
# Paths
# -------------------------
FRAME_DIR = "./data/2_frame_files"
OCR_DIR = "./data/3_ocr_results"


from concurrent.futures import ProcessPoolExecutor, as_completed


# -------------------------
# 데이터 인덱스 구축
# -------------------------
NUM_INDEX_WORKERS = 16


def _index_channel(channel):
    """채널 하나의 영상 목록 반환"""
    ocr_channel_dir = os.path.join(OCR_DIR, channel)
    frame_channel_dir = os.path.join(FRAME_DIR, channel)
    results = []
    for pf in sorted(os.listdir(ocr_channel_dir)):
        if not pf.endswith(".parquet"):
            continue
        video_name = pf[:-8]
        video_frame_dir = os.path.join(frame_channel_dir, video_name)
        if not os.path.isdir(video_frame_dir):
            continue
        results.append((channel, video_name, os.path.join(ocr_channel_dir, pf), video_frame_dir))
    return results


def build_index():
    """(channel, video, parquet_path, frame_dir) 목록 생성 (병렬)"""
    channels = sorted([d for d in os.listdir(OCR_DIR) if os.path.isdir(os.path.join(OCR_DIR, d))])
    index = []
    with ProcessPoolExecutor(max_workers=NUM_INDEX_WORKERS) as executor:
        futures = {executor.submit(_index_channel, ch): ch for ch in channels}
        for future in tqdm(as_completed(futures), total=len(channels), desc="인덱스 구축"):
            index.extend(future.result())
    return index


# -------------------------
# xywha → 4개 꼭짓점
# -------------------------
def xywha_to_corners(cx, cy, w, h, angle_deg):
    angle = math.radians(angle_deg)
    cos_a, sin_a = math.cos(angle), math.sin(angle)
    dx_w, dy_w = cos_a * w / 2, sin_a * w / 2
    dx_h, dy_h = -sin_a * h / 2, cos_a * h / 2
    return np.array([
        [cx - dx_w - dx_h, cy - dy_w - dy_h],
        [cx + dx_w - dx_h, cy + dy_w - dy_h],
        [cx + dx_w + dx_h, cy + dy_w + dy_h],
        [cx - dx_w + dx_h, cy - dy_w + dy_h],
    ], dtype=np.int32)


# -------------------------
# 시각화
# -------------------------
def visualize(score_min, score_max, index):
    if not index:
        return None, "인덱스가 비어있습니다."

    # 최대 50번 시도
    import polars as pl

    for _ in range(50):
        channel, video, parquet_path, frame_dir = random.choice(index)
        df = pl.read_parquet(parquet_path, glob=False)

        # 스코어 범위에 맞는 프레임 필터
        candidates = []
        for row in df.iter_rows(named=True):
            scores = json.loads(row["ocr_scores"])
            texts = json.loads(row["ocr_texts"])
            xywhas = json.loads(row["ocr_xywha"])
            matched = [
                (t, s, x) for t, s, x in zip(texts, scores, xywhas)
                if score_min <= s <= score_max
            ]
            if matched:
                candidates.append((row["frame_num"], matched))

        if not candidates:
            continue

        frame_num, matched_items = random.choice(candidates)
        img_path = os.path.join(frame_dir, f"frame_{frame_num:08d}.jpg")
        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        # bbox 그리기
        for text, score, xywha in matched_items:
            cx, cy, w, h, angle = xywha
            corners = xywha_to_corners(cx, cy, w, h, angle)
            cv2.polylines(img, [corners], isClosed=True, color=(0, 255, 0), thickness=2)
            label = f"{text} ({score:.2f})"
            lx, ly = int(corners[0][0]), int(corners[0][1]) - 5
            cv2.putText(img, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        # 범위 밖 텍스트도 회색으로 표시
        all_scores = json.loads(df.filter(pl.col("frame_num") == frame_num).row(0, named=True)["ocr_scores"])
        all_texts = json.loads(df.filter(pl.col("frame_num") == frame_num).row(0, named=True)["ocr_texts"])
        all_xywhas = json.loads(df.filter(pl.col("frame_num") == frame_num).row(0, named=True)["ocr_xywha"])
        for text, score, xywha in zip(all_texts, all_scores, all_xywhas):
            if not (score_min <= score <= score_max):
                cx, cy, w, h, angle = xywha
                corners = xywha_to_corners(cx, cy, w, h, angle)
                cv2.polylines(img, [corners], isClosed=True, color=(128, 128, 128), thickness=1)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 매칭 텍스트 상세
        matched_lines = [f"  [{s:.4f}] {t}" for t, s, x in matched_items]
        # 범위 밖 텍스트 상세
        outside_items = [(t, s) for t, s, x in zip(all_texts, all_scores, all_xywhas)
                         if not (score_min <= s <= score_max)]
        outside_lines = [f"  [{s:.4f}] {t}" for t, s in outside_items]

        info = (
            f"채널: {channel}\n"
            f"영상: {video}\n"
            f"프레임: {frame_num}\n"
            f"\n── 매칭 (초록) {len(matched_items)}개 ──\n"
            + ("\n".join(matched_lines) if matched_lines else "  없음")
            + f"\n\n── 범위 밖 (회색) {len(outside_items)}개 ──\n"
            + ("\n".join(outside_lines) if outside_lines else "  없음")
        )
        return img_rgb, info

    return None, "조건에 맞는 프레임을 찾지 못했습니다."


# -------------------------
# Gradio UI
# -------------------------
def main():
    print("📁 인덱스 구축 중...")
    index = build_index()
    print(f"✅ {len(index)}개 영상 인덱스 완료")

    def on_click(score_min, score_max):
        return visualize(score_min, score_max, index)

    with gr.Blocks(title="OCR Score EDA") as demo:
        gr.Markdown("# OCR Score EDA\n스코어 범위를 조절하고 랜덤 샘플을 확인하세요.")

        with gr.Row():
            score_min = gr.Slider(0.0, 1.0, value=0.0, step=0.01, label="최소 스코어")
            score_max = gr.Slider(0.0, 1.0, value=1.0, step=0.01, label="최대 스코어")

        btn = gr.Button("랜덤 샘플 보기", variant="primary")

        with gr.Row():
            with gr.Column(scale=9):
                image_out = gr.Image(label="프레임", height=800)
            with gr.Column(scale=7):
                info_out = gr.Textbox(label="정보", lines=20)

        btn.click(fn=on_click, inputs=[score_min, score_max], outputs=[image_out, info_out])

    demo.launch(server_name="0.0.0.0", server_port=7860)


if __name__ == "__main__":
    main()

📁 인덱스 구축 중...


인덱스 구축: 100%|██████████| 664/664 [00:00<00:00, 4713.70it/s]


✅ 66400개 영상 인덱스 완료
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.
